In [ ]:
import json

import pandas as pd

with open("2024學年度_10411-03-01-2_臺中市高級中等學校外國學生數.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
df = pd.DataFrame(data=data)

In [ ]:
df.head(5)

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.head(5)

In [ ]:
df.to_csv("測試.csv")

In [ ]:
from langchain_core.prompts.image import ImagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, PromptTemplate


def build_standard_chat_prompt_template(kwargs):
    """Build a ChatPromptTemplate from system and human message configs.

    Supports both text-only and multimodal (text + image) prompts.

    Args:
        kwargs: Dictionary with optional 'system' and 'human' keys.
            Each value is a dict with 'template' (required) and optional
            'type' ('text' or 'image') and 'input_variables'.

    Returns:
        A configured ChatPromptTemplate ready for use with an LLM.
    """
    messages = []

    if 'system' in kwargs:
        content = kwargs.get('system')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = [PromptTemplate(**c) for c in content]
        else:
            prompts = [PromptTemplate(**content)]

        message = SystemMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    if 'human' in kwargs:
        content = kwargs.get('human')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = []
            for c in content:
                if c.get("type") == "image":
                    prompts.append(ImagePromptTemplate(**c))
                else:
                    prompts.append(PromptTemplate(**c))
        else:
            if content.get("type") == "image":
                prompts = [ImagePromptTemplate(**content)]
            else:
                prompts = [PromptTemplate(**content)]

        message = HumanMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    chat_prompt_template = ChatPromptTemplate.from_messages(messages)

    return chat_prompt_template


In [ ]:
df.to_csv("測試.csv")

In [ ]:
from langchain_core.prompts.image import ImagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, PromptTemplate


def build_standard_chat_prompt_template(kwargs):
    """Build a ChatPromptTemplate from system and human message configs.

    Supports both text-only and multimodal (text + image) prompts.

    Args:
        kwargs: Dictionary with optional 'system' and 'human' keys.
            Each value is a dict with 'template' (required) and optional
            'type' ('text' or 'image') and 'input_variables'.

    Returns:
        A configured ChatPromptTemplate ready for use with an LLM.
    """
    messages = []

    if 'system' in kwargs:
        content = kwargs.get('system')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = [PromptTemplate(**c) for c in content]
        else:
            prompts = [PromptTemplate(**content)]

        message = SystemMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    if 'human' in kwargs:
        content = kwargs.get('human')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = []
            for c in content:
                if c.get("type") == "image":
                    prompts.append(ImagePromptTemplate(**c))
                else:
                    prompts.append(PromptTemplate(**c))
        else:
            if content.get("type") == "image":
                prompts = [ImagePromptTemplate(**content)]
            else:
                prompts = [PromptTemplate(**content)]

        message = HumanMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    chat_prompt_template = ChatPromptTemplate.from_messages(messages)

    return chat_prompt_template


建立工具: Schema Tool

In [ ]:
import io
import os
from contextlib import redirect_stdout
from pydantic import BaseModel, Field
from textwrap import dedent


from langchain.tools import BaseTool, ToolRuntime
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import Runnable


class Context(BaseModel):
    directory: str


SCHEMA_SYSTEM_PROMPT = dedent("""
# Role
你是一位專業的 Python 資料分析師，精通 pandas 資料處理。你的任務是為指定的 CSV 檔案生成精確、可執行的 Python 代碼。

# Goal
生成一段可直接執行的 Python 代碼，使用 pandas 讀取 CSV 檔案，並輸出該檔案的結構摘要與前五筆資料。

# Input
- <file>: 待分析的 CSV 檔案完整路徑

# Rule
- 使用 `pd.read_csv()` 讀取 CSV 檔案，指定 `encoding='utf-8'`
- 使用 `df.info()` 輸出人類可讀的完整摘要（欄位名、非空值數量、資料型別）
- 使用 `print(df.head(5).to_string())` 輸出前五筆資料
- 輸出的代碼必須是純 Python 代碼，不含任何 markdown 標記或解釋文字

# Constraints
- 嚴禁輸出 markdown 代碼塊標記（如 ```python 或 ```）
- 嚴禁在代碼前後添加任何說明、註解或對話文字
- 嚴禁使用任何未安裝的第三方套件（僅限 pandas 與 Python 標準庫）
- 嚴禁修改、刪除或寫入任何檔案
- 嚴禁使用網路請求或外部 API

# Reasoning (Chain of Thought)
請依以下步驟逐步推理，每完成一步再進行下一步：

Step 1: [狀態確認] 確認輸入的 CSV 檔案路徑 {file}，判斷是否需要特殊編碼處理
Step 2: [關鍵分析] 確定讀取策略：使用 pd.read_csv()，設定適當的 encoding 參數
Step 3: [推理展開] 構建輸出邏輯：先呼叫 df.info() 輸出結構摘要，再呼叫 print(df.head(5).to_string()) 輸出前五筆
Step 4: [驗證檢查] 檢查代碼是否僅包含必要的 import 與執行語句，無多餘內容
Step 5: [整合輸出] 輸出純 Python 代碼字串，不含任何格式包裝
""")


class SchemaTool(BaseTool):
    name: str = "schema_tool"
    description: str = dedent("""
    Reads all files in a given directory and returns the schema (structure summary) and first 5 rows of each file. Uses pandas to analyze CSV files, providing column names, data types, non-null counts, and sample data. Use this tool when you need to understand the structure and content of data files before further processing.
    """)

    pipeline: Runnable

    @classmethod
    def create(cls, llm: Runnable):

        input_ = {
            "system": {"template": SCHEMA_SYSTEM_PROMPT},
            "human": {
                "template": dedent("""
                    <file>: {file}
                """),
                "input_variable": ["file"]
            }
        }
        pipeline = build_standard_chat_prompt_template(input_) | llm | StrOutputParser()

        return cls(pipeline=pipeline)

    def _run(self, runtime: ToolRuntime[Context], **input):
        print(f"Running {self.name}...")
        directory = runtime.context.directory
    
        raw_output = ""
        
        for f in os.listdir(directory):
            
            if not f.endswith(".csv"):
                continue

            print(f)
            
            file = os.path.join(directory, f)
            
            code = self.pipeline.invoke({
                "file": file
            })
    
            stdout_capture = io.StringIO()
            try:
                with redirect_stdout(stdout_capture):
                    exec(code, {"__builtins__": __builtins__})
                output = stdout_capture.getvalue()
            except Exception as e:
                output = f"EXECUTION ERROR: {str(e)}"

            raw_output += f"-{file}: {output}\n\n"
            
        return raw_output

    async def _arun(self, runtime: ToolRuntime[Context]):

        return "Not implemented Yet"

In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.agents.middleware.tool_retry import ToolRetryMiddleware
from langchain.agents.middleware.tool_call_limit import ToolCallLimitMiddleware
from langchain_core.messages import HumanMessage


AGENT_INSTRUCTION = dedent("""
# Role
你是一位專業的數據分析顧問，擅長解讀資料結構並提供清晰的分析建議。你的工作方式是先了解資料的樣貌，再根據使用者需求給出精準的回應。

# Goal
協助使用者理解資料內容並回答數據分析相關問題。

# Tool
你可以使用以下工具來完成任務：

- **schema_tool**: 讀取指定目錄中的所有 CSV 檔案，返回每個檔案的結構摘要（欄位名稱、資料型別、非空值數量）與前五筆範例資料。適用於需要了解資料結構與內容的場景。

# Input
使用者會以自然語言描述他們的數據分析需求或問題。

# Rule
- 當使用者詢問關於資料檔案的內容、結構或分析建議時，先調用工具獲取資料資訊
- 根據工具返回的結構摘要與前五筆資料，解讀每個欄位的意義與資料型別
- 針對使用者的具體問題，基於實際的資料結構給出分析建議或答案
- 若工具返回錯誤，如實告知使用者並建議檢查檔案格式

# Constraints
- 嚴禁在未調用工具的情況下憑空猜測資料內容
- 嚴禁對資料進行任何寫入、修改或刪除操作
- 回答必須基於工具返回的實際資料，不得虛構

# Reasoning (ReAct)
使用 ReAct（Reasoning + Acting）框架，在「推理」與「行動」之間交替進行：

Thought: 分析使用者需求，判斷是否需要調用工具，以及需要哪些資訊
Action: 選擇並調用合適的工具，傳入必要參數
Observation: 仔細觀察工具返回的結果，提取關鍵資訊
Thought: 根據觀察結果進行分析推理，判斷是否需要進一步行動
...（重複 Thought → Action → Observation 循環，直到足以回答問題）
Final Answer: 給出清晰、有條理的最終回答
""")


llm = ChatOllama(model='deepseek-v4-pro:cloud',
                 base_url='https://ollama.com',
                 name='main', temperature=0)


tools = [SchemaTool.create(llm=llm)]

agent = create_agent(
    model=llm,
    name="analysis_agent",
    tools=tools,
    system_prompt=AGENT_INSTRUCTION,
    middleware=[
        ToolRetryMiddleware(max_retries=2),
        ToolCallLimitMiddleware(exit_behavior='end',
                                run_limit=1,
                                tool_name="schema_tool")
    ]
)

In [ ]:
context = Context(directory="./")

In [ ]:
agent_input = {"messages": HumanMessage(content="顯示所有檔案的schema")}
agent_result = agent.invoke(agent_input, context=context)

In [ ]:
agent_result

In [ ]:
print(agent_result["messages"][-1].content)

Pandas Tool

In [ ]:
from typing import List


PANDAS_SYSTEM_PROMPT = dedent("""
# Role
你是一位專業的 Python 資料科學家，精通 pandas、scikit-learn、scipy 等資料處理與科學計算庫。你的任務是根據使用者指定的 CSV 檔案，生成精確、可執行的 Python 代碼來進行數據處理與分析。

# Goal
生成一段可直接執行的 Python 代碼，使用 pandas、scikit-learn、scipy 等專業庫對指定的 CSV 檔案進行數據處理與分析，並輸出處理結果。

# Input
- <files>: 待處理的 CSV 檔案完整路徑列表
- <context>: 用戶的需求

# Rule
- 使用 `pd.read_csv()` 讀取每個 CSV 檔案，指定 `encoding='utf-8'`
- 根據需求選用合適的庫進行數據處理與分析：
  - **pandas**: 資料讀取、篩選、分組、聚合、合併（`pd.merge` / `pd.concat`）、排序、描述性統計
  - **scikit-learn**: 資料預處理（標準化、編碼、降維）、模型訓練與預測、評估指標
  - **scipy**: 統計檢定（t-test、chi-square、ANOVA）、優化、信號處理、稀疏矩陣運算
- 使用 `print()` 輸出處理結果，確保輸出清晰可讀
- 輸出的代碼必須是純 Python 代碼，不含任何 markdown 標記或解釋文字

# Constraints
- 嚴禁輸出 markdown 代碼塊標記（如 ```python 或 ```）
- 嚴禁在代碼前後添加任何說明、註解或對話文字
- 僅限使用 pandas、scikit-learn、scipy、numpy 與 Python 標準庫，嚴禁使用其他未安裝的第三方套件
- 嚴禁修改、刪除或寫入任何檔案
- 嚴禁使用網路請求或外部 API

# Reasoning (Chain of Thought)
請依以下步驟逐步推理，每完成一步再進行下一步：

Step 1: [狀態確認] 確認輸入的 CSV 檔案列表 {files}，判斷檔案數量、結構與可能的關聯性
Step 2: [需求分析] 根據資料特性與分析目標，選擇最合適的庫與方法（pandas 處理 / sklearn 建模 / scipy 檢定）
Step 3: [推理展開] 構建處理流程：讀取 → 預處理 → 分析/建模 → 輸出結果
Step 4: [驗證檢查] 檢查代碼是否僅包含必要的 import 與執行語句，無多餘內容
Step 5: [整合輸出] 輸出純 Python 代碼字串，不含任何格式包裝
""")


class FilesInputs(BaseModel):
    files: List[str] = Field(description="List of CSV file names to process. Provide the filenames (without directory path) of the data files you want to analyze or transform.")
    context: str = Field(description="Summary of the schema of the files")

class PandasTool(BaseTool):
    name: str = "pandas_tool"

    description_template: str = dedent("""
Processes specified CSV files using Python and pandas. Provide a list of file names to perform data operations such as filtering, grouping, aggregation, merging, sorting, and statistical analysis. Use this tool when you need to transform, analyze, or compute results from data files.

{input_format_instructions}
    """)

    input_parser: PydanticOutputParser = PydanticOutputParser(pydantic_object=FilesInputs)
    input_format_instructions: str = input_parser.get_format_instructions()
    
    description: str = description_template.format(input_format_instructions=input_format_instructions)

    pipeline: Runnable

    @classmethod
    def create(cls, llm: Runnable):
        
        input_ = {
            "system": {"template": PANDAS_SYSTEM_PROMPT},
            "human": {
                "template": dedent("""
                    <files>: {files},
                    <context>: {context}
                """),
                "input_variable": ["files", "context"]
            }
        }
        pipeline = build_standard_chat_prompt_template(input_) | llm | StrOutputParser()

        return cls(pipeline=pipeline)

    def _run(self, runtime: ToolRuntime[Context], **input):
        print(f"Running {self.name}...")
        directory = runtime.context.directory

        args = input.get("input", input)

        files = args['files']
        context = args['context']

        print("="*20)
        print(context)
        print("="*20)
        
        code = self.pipeline.invoke({
            "files": [os.path.join(directory, f) for f in files],
            "context": context
        })

        print("="*20)
        print("code:")
        print(code)
        print("="*20)
        
        stdout_capture = io.StringIO()
        try:
            with redirect_stdout(stdout_capture):
                exec(code, {"__builtins__": __builtins__})
            output = stdout_capture.getvalue()
        except Exception as e:
            output = f"EXECUTION ERROR: {str(e)}"
            
        return output

    async def _arun(self, runtime: ToolRuntime[Context]):

        return "Not implemented Yet"

In [ ]:
AGENT_INSTRUCTION_VERSION_2 = dedent("""
# Role
你是一位專業的數據分析顧問，擅長解讀資料結構並提供清晰的分析建議。你的工作方式是先了解資料的樣貌，再根據使用者需求給出精準的回應。

# Goal
協助使用者理解資料內容並回答數據分析相關問題。

# Tool
你可以使用以下工具來完成任務：

- **schema_tool**: 讀取指定目錄中的所有 CSV 檔案，返回每個檔案的結構摘要（欄位名稱、資料型別、非空值數量）與前五筆範例資料。注意：此工具僅提供結構資訊與極少量範例，無法用於統計、篩選或計算。
- **pandas_tool**: 使用 Python 與 pandas、scikit-learn、scipy 等專業庫對指定的 CSV 檔案進行數據處理與分析。支援篩選、分組、聚合、合併、統計檢定、機器學習建模等操作。這是唯一能對資料進行實際計算與分析的工具。

# Input
使用者會以自然語言描述他們的數據分析需求或問題。

# Rule
- 第一步：調用 `schema_tool` 了解有哪些檔案、每個檔案的欄位結構與資料型別
- 第二步：根據 schema_tool 返回的欄位資訊，調用 `pandas_tool` 對相關檔案進行實際的數據處理與分析
- 重要：schema_tool 僅提供結構與 5 筆範例，任何涉及篩選、統計、聚合、計數、排序、建模的操作都必須透過 pandas_tool 完成
- 調用 pandas_tool 時，從 schema_tool 的結果中選取相關的檔案名稱傳入
- 若工具返回錯誤，如實告知使用者並建議檢查檔案格式或需求描述

# Constraints
- 嚴禁在未調用工具的情況下憑空猜測資料內容
- 嚴禁僅憑 schema_tool 的 5 筆範例資料就回答需要統計或計算的問題
- 嚴禁對資料進行任何寫入、修改或刪除操作
- 回答必須基於 pandas_tool 的實際計算結果，不得虛構

# Reasoning (ReAct)
使用 ReAct（Reasoning + Acting）框架，在「推理」與「行動」之間交替進行：

Thought: 分析使用者需求，判斷這是否為純結構問題（只需 schema_tool）還是數據分析問題（需要 pandas_tool 進行計算）
Action: 若需要了解結構，調用 schema_tool；若需要計算分析，調用 pandas_tool
Observation: 仔細觀察工具返回的結果，提取關鍵資訊
Thought: 若已調用 schema_tool 但問題需要數據計算，則必須繼續調用 pandas_tool，不可在此停止
...（重複 Thought → Action → Observation 循環，直到足以回答問題）
Final Answer: 基於 pandas_tool 的計算結果，給出清晰、有條理的最終回答
""")


llm = ChatOllama(model='deepseek-v4-pro:cloud',
                 base_url='https://ollama.com',
                 name='main', temperature=0)


tools = [SchemaTool.create(llm=llm),
         PandasTool.create(llm=llm)]

agent = create_agent(
    model=llm,
    name="analysis_agent_version_2",
    tools=tools,
    system_prompt=AGENT_INSTRUCTION_VERSION_2,
    middleware=[
        ToolRetryMiddleware(max_retries=2),
    ]
)

In [ ]:
agent_input = {"messages": HumanMessage(content="顯示外國學生數量")}
agent_result = agent.invoke(agent_input, context=context)

In [ ]:
print(agent_result['messages'][-1].content)

In [ ]:
df["數值"].astype(int).sum()

In [ ]:
print(agent_result['messages'][-1].content)

In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.agents.middleware.tool_retry import ToolRetryMiddleware
from langchain.agents.middleware.tool_call_limit import ToolCallLimitMiddleware
from langchain_core.messages import HumanMessage


AGENT_INSTRUCTION = dedent("""
# Role
你是一位專業的數據分析顧問，擅長解讀資料結構並提供清晰的分析建議。你的工作方式是先了解資料的樣貌，再根據使用者需求給出精準的回應。

# Goal
協助使用者理解資料內容並回答數據分析相關問題。

# Tool
你可以使用以下工具來完成任務：

- **schema_tool**: 讀取指定目錄中的所有 CSV 檔案，返回每個檔案的結構摘要（欄位名稱、資料型別、非空值數量）與前五筆範例資料。適用於需要了解資料結構與內容的場景。

# Input
使用者會以自然語言描述他們的數據分析需求或問題。

# Rule
- 當使用者詢問關於資料檔案的內容、結構或分析建議時，先調用工具獲取資料資訊
- 根據工具返回的結構摘要與前五筆資料，解讀每個欄位的意義與資料型別
- 針對使用者的具體問題，基於實際的資料結構給出分析建議或答案
- 若工具返回錯誤，如實告知使用者並建議檢查檔案格式

# Constraints
- 嚴禁在未調用工具的情況下憑空猜測資料內容
- 嚴禁對資料進行任何寫入、修改或刪除操作
- 回答必須基於工具返回的實際資料，不得虛構

# Reasoning (ReAct)
使用 ReAct（Reasoning + Acting）框架，在「推理」與「行動」之間交替進行：

Thought: 分析使用者需求，判斷是否需要調用工具，以及需要哪些資訊
Action: 選擇並調用合適的工具，傳入必要參數
Observation: 仔細觀察工具返回的結果，提取關鍵資訊
Thought: 根據觀察結果進行分析推理，判斷是否需要進一步行動
...（重複 Thought → Action → Observation 循環，直到足以回答問題）
Final Answer: 給出清晰、有條理的最終回答
""")


llm = ChatOllama(model='deepseek-v4-pro:cloud',
                 base_url='https://ollama.com',
                 name='main', temperature=0)


tools = [SchemaTool.create(llm=llm)]

agent = create_agent(
    model=llm,
    name="analysis_agent",
    tools=tools,
    system_prompt=AGENT_INSTRUCTION,
    middleware=[
        ToolRetryMiddleware(max_retries=2),
        ToolCallLimitMiddleware(exit_behavior='end',
                                run_limit=1,
                                tool_name="schema_tool")
    ]
)

In [ ]:
context = Context(directory="./")

In [ ]:
agent_input = {"messages": HumanMessage(content="顯示所有檔案的schema")}
agent_result = agent.invoke(agent_input, context=context)

In [ ]:
agent_result

In [ ]:
print(agent_result["messages"][-1].content)

Pandas Tool

In [ ]:
from typing import List


PANDAS_SYSTEM_PROMPT = dedent("""
# Role
你是一位專業的 Python 資料科學家，精通 pandas、scikit-learn、scipy 等資料處理與科學計算庫。你的任務是根據使用者指定的 CSV 檔案，生成精確、可執行的 Python 代碼來進行數據處理與分析。

# Goal
生成一段可直接執行的 Python 代碼，使用 pandas、scikit-learn、scipy 等專業庫對指定的 CSV 檔案進行數據處理與分析，並輸出處理結果。

# Input
- {files}: 待處理的 CSV 檔案完整路徑列表

# Rule
- 使用 `pd.read_csv()` 讀取每個 CSV 檔案，指定 `encoding='utf-8'`
- 根據需求選用合適的庫進行數據處理與分析：
  - **pandas**: 資料讀取、篩選、分組、聚合、合併（`pd.merge` / `pd.concat`）、排序、描述性統計
  - **scikit-learn**: 資料預處理（標準化、編碼、降維）、模型訓練與預測、評估指標
  - **scipy**: 統計檢定（t-test、chi-square、ANOVA）、優化、信號處理、稀疏矩陣運算
- 使用 `print()` 輸出處理結果，確保輸出清晰可讀
- 輸出的代碼必須是純 Python 代碼，不含任何 markdown 標記或解釋文字

# Constraints
- 嚴禁輸出 markdown 代碼塊標記（如 ```python 或 ```）
- 嚴禁在代碼前後添加任何說明、註解或對話文字
- 僅限使用 pandas、scikit-learn、scipy、numpy 與 Python 標準庫，嚴禁使用其他未安裝的第三方套件
- 嚴禁修改、刪除或寫入任何檔案
- 嚴禁使用網路請求或外部 API

# Reasoning (Chain of Thought)
請依以下步驟逐步推理，每完成一步再進行下一步：

Step 1: [狀態確認] 確認輸入的 CSV 檔案列表 {files}，判斷檔案數量、結構與可能的關聯性
Step 2: [需求分析] 根據資料特性與分析目標，選擇最合適的庫與方法（pandas 處理 / sklearn 建模 / scipy 檢定）
Step 3: [推理展開] 構建處理流程：讀取 → 預處理 → 分析/建模 → 輸出結果
Step 4: [驗證檢查] 檢查代碼是否僅包含必要的 import 與執行語句，無多餘內容
Step 5: [整合輸出] 輸出純 Python 代碼字串，不含任何格式包裝
""")


class FilesInputs(BaseModel):
    files: List[str] = Field(description="List of CSV file names to process. Provide the filenames (without directory path) of the data files you want to analyze or transform.")


class PandasTool(BaseTool):
    name: str = "pandas_tool"

    description_template: str = dedent("""
Processes specified CSV files using Python and pandas. Provide a list of file names to perform data operations such as filtering, grouping, aggregation, merging, sorting, and statistical analysis. Use this tool when you need to transform, analyze, or compute results from data files.

{input_format_instructions}
    """)

    input_parser: PydanticOutputParser = PydanticOutputParser(pydantic_object=FilesInputs)
    input_format_instructions: str = input_parser.get_format_instructions()
    
    description: str = description_template.format(input_format_instructions=input_format_instructions)

    pipeline: Runnable

    @classmethod
    def create(cls, llm: Runnable):
        
        input_ = {
            "system": {"template": PANDAS_SYSTEM_PROMPT},
            "human": {
                "template": dedent("""
                    <files>: {files}
                """),
                "input_variable": ["files"]
            }
        }
        pipeline = build_standard_chat_prompt_template(input_) | llm | StrOutputParser()

        return cls(pipeline=pipeline)

    def _run(self, runtime: ToolRuntime[Context], **input):
        print(f"Running {self.name}...")
        directory = runtime.context.directory

        args = input.get("input", input)

        files = args['files']
        
        code = self.pipeline.invoke({
            "files": [os.path.join(directory, f) for f in files]
        })

        stdout_capture = io.StringIO()
        try:
            with redirect_stdout(stdout_capture):
                exec(code, {"__builtins__": __builtins__})
            output = stdout_capture.getvalue()
        except Exception as e:
            output = f"EXECUTION ERROR: {str(e)}"
            
        return output

    async def _arun(self, runtime: ToolRuntime[Context]):

        return "Not implemented Yet"

In [ ]:
AGENT_INSTRUCTION_VERSION_2 = dedent("""
# Role
你是一位專業的數據分析顧問，擅長解讀資料結構並提供清晰的分析建議。你的工作方式是先了解資料的樣貌，再根據使用者需求給出精準的回應。

# Goal
協助使用者理解資料內容並回答數據分析相關問題。

# Tool
你可以使用以下工具來完成任務：

- **schema_tool**: 讀取指定目錄中的所有 CSV 檔案，返回每個檔案的結構摘要（欄位名稱、資料型別、非空值數量）與前五筆範例資料。注意：此工具僅提供結構資訊與極少量範例，無法用於統計、篩選或計算。
- **pandas_tool**: 使用 Python 與 pandas、scikit-learn、scipy 等專業庫對指定的 CSV 檔案進行數據處理與分析。支援篩選、分組、聚合、合併、統計檢定、機器學習建模等操作。這是唯一能對資料進行實際計算與分析的工具。

# Input
使用者會以自然語言描述他們的數據分析需求或問題。

# Rule
- 第一步：調用 `schema_tool` 了解有哪些檔案、每個檔案的欄位結構與資料型別
- 第二步：根據 schema_tool 返回的欄位資訊，調用 `pandas_tool` 對相關檔案進行實際的數據處理與分析
- 重要：schema_tool 僅提供結構與 5 筆範例，任何涉及篩選、統計、聚合、計數、排序、建模的操作都必須透過 pandas_tool 完成
- 調用 pandas_tool 時，從 schema_tool 的結果中選取相關的檔案名稱傳入
- 若工具返回錯誤，如實告知使用者並建議檢查檔案格式或需求描述

# Constraints
- 嚴禁在未調用工具的情況下憑空猜測資料內容
- 嚴禁僅憑 schema_tool 的 5 筆範例資料就回答需要統計或計算的問題
- 嚴禁對資料進行任何寫入、修改或刪除操作
- 回答必須基於 pandas_tool 的實際計算結果，不得虛構

# 推理與行動（ReAct）
使用「推理 → 行動 → 觀察」的循環方式工作，每一步都先思考再決定是否呼叫工具：

- **推理**：分析使用者需求與目前已取得的資料，判斷下一步該做什麼。問自己：「我現在擁有的資訊是否已經足以完整回答使用者的問題？」
- **行動**：若資訊不足，呼叫對應的工具（schema_tool 或 pandas_tool）來獲取更多資料
- **觀察**：仔細閱讀工具回傳的結果，提取關鍵資訊，然後回到推理步驟

## 停止條件
當你判斷已擁有足夠的資訊能完整回答使用者問題時，**立即停止呼叫工具**，直接輸出最終答案。最終答案應包含：
- 清晰的數據解讀
- 具體的數字或統計結果
- 必要時提供趨勢分析或比較

切記：一旦能回答問題就停止，不要過度分析或呼叫不必要的工具。
""")


llm = ChatOllama(model='deepseek-v4-pro:cloud',
                 base_url='https://ollama.com',
                 name='main', temperature=0)


tools = [SchemaTool.create(llm=llm),
         PandasTool.create(llm=llm)]

agent = create_agent(
    model=llm,
    name="analysis_agent_version_2",
    tools=tools,
    system_prompt=AGENT_INSTRUCTION_VERSION_2,
    middleware=[
        ToolRetryMiddleware(max_retries=2),
    ]
)

In [ ]:
agent_input = {"messages": HumanMessage(content="顯示外國語學校的學生數量")}
agent_result = agent.invoke(agent_input, context=context)

In [ ]:
print(agent_result['messages'][-1].content)

In [ ]:
df["數值"].astype(int).sum()

In [ ]:
print(agent_result['messages'][-1].content)

跨文檔工作

In [ ]:
from pathlib import Path

directory = "多檔案測試"

for file in os.listdir(directory):

    path = Path(file)
    
    if not file.endswith("json"):
        continue
    with open(os.path.join(directory, file), "r", encoding="utf-8") as f:
        data = json.load(f)
        df = pd.DataFrame(data=data)
        df.to_csv(os.path.join(directory, f"{path.stem}.csv"))

In [ ]:
context = Context(directory=directory)

agent = create_agent(
    model=llm,
    name="analysis_agent_version_2",
    tools=tools,
    system_prompt=AGENT_INSTRUCTION_VERSION_2,
    middleware=[
        ToolRetryMiddleware(max_retries=2),
    ]
)

In [ ]:
agent_input = {"messages": HumanMessage(content="台中市學生從2021年到2024年的變化")}
agent_result = agent.invoke(agent_input, context=context)